| Step | Roadmap name                     | Main notebook section            | Main output                       |   |     |   |     |
| ---- | -------------------------------- | -------------------------------- | --------------------------------- | - | --- | - | --- |
| 1    | **Dataset Preparation**          | `1. Dataset Preparation`         | Clean \(X\), labels               |   |     |   |     |
| 2    | **Data Understanding**           | `2. Data Understanding`          | Dimensions, class information     |   |     |   |     |
| 3    | **Z-Score Normalization**        | `3. Data Normalization`          | Normalized \(X\)                  |   |     |   |     |
| 4    | **Laplacian Graph Construction** | `4. Manifold Structure Learning` | \(A,D,L_X\)                       |   |     |   |     |
| 5    | **Manifold Preservation**        | `5. Manifold Regularization`     | Understand Eq. (1)                |   |     |   |     |
| 6    | **Autoencoder Architecture**     | `6. Sparse Manifold Autoencoder` | \(W^{(1)},W^{(2)},Z,\bar X\)      |   |     |   |     |
| 7    | **Objective Function**           | `7. Optimization Objective`      | Eq. (5)                           |   |     |   |     |
| 8    | **Parameter Initialization**     | `8. Initialization`              | Initial \(W^{(1)},W^{(2)},H,Y,Q\) |   |     |   |     |
| 9    | **Weight Updates**               | `9. Optimization`                | Updated weights                   |   |     |   |     |
| 10   | **H Update**                     | `10. Auxiliary Graph Update`     | Updated \(H\)                     |   |     |   |     |
| 11   | **Training**                     | `11. Model Training`             | Convergence/objective             |   |     |   |     |
| 12   | **Feature Importance**           | `12. Feature Ranking`            | (      w_i_2) |
| 13   | **Feature Selection**            | `13. Feature Selection`          | Top 50–300 genes                  |   |     |   |     |
| 14   | **Clustering**                   | `14. Clustering Evaluation`      | K-means clusters                  |   |     |   |     |
| 15   | **Evaluation**                   | `15. ACC and NMI`                | ACC, NMI                          |   |     |   |     |
| 16   | **Paper Comparison**             | `16. Comparison with Paper`      | Colon: paper vs ours              |   |     |   |     |
| 17   | **Ablation**                     | `17. Ablation Study`             | With/without \(H\)                |   |     |   |     |
| 18   | **Convergence Analysis**         | `18. Convergence Analysis`       | Objective curve                   |   |     |   |     |
| 19   | **Final Results**                | `19. Final Results`              | CSV + tables                      |   |     |   |     |
| 20   | **Thesis Interpretation**        | `20. Discussion`                 | What we learned/found             |   |     |   |     |


### Data set Prepration

#### Import Libraries

In [2]:
import numpy as np
import pandas as pd

#### Load Colon Dataset

In [3]:
data = pd.read_csv("../data/preprocessed/colon.csv")

print("Dataset shape:", data.shape)
display(data.head())

Dataset shape: (62, 2001)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_1992,feature_1993,feature_1994,feature_1995,feature_1996,feature_1997,feature_1998,feature_1999,feature_2000,label
0,2,0,0,0,-2,0,-2,0,2,0,...,0,0,0,0,-2,0,0,2,-2,-1
1,2,2,0,0,-2,0,0,0,2,0,...,0,-2,0,-2,-2,2,2,0,-2,1
2,-2,2,2,0,-2,-2,-2,-2,-2,-2,...,0,-2,2,-2,0,-2,-2,-2,-2,-1
3,0,2,2,0,-2,-2,-2,-2,0,0,...,-2,-2,2,-2,-2,0,-2,0,-2,1
4,-2,-2,0,0,-2,-2,-2,0,-2,0,...,0,-2,-2,-2,0,0,-2,-2,0,-1


### Data Understanding

#### Separate Features and Labels

In [4]:
X = data.drop(columns=["label"])
y = data["label"]

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

Feature matrix shape: (62, 2000)
Label vector shape: (62,)


#### Check Classes

In [5]:
print("Classes:", sorted(y.unique()))
print("Class distribution:")
print(y.value_counts())

Classes: [np.int64(-1), np.int64(1)]
Class distribution:
label
-1    40
 1    22
Name: count, dtype: int64


#### Check Missing Values

In [6]:
print("Missing values:", X.isnull().sum().sum())

Missing values: 0


#### Convert Feature Matrix to NumPy

In [7]:
X = X.to_numpy(dtype=float)
y = y.to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (62, 2000)
y shape: (62,)


### Data Normalization

Paper menitoned about zscore normalization of features

In [8]:
# Calculate mean and standard deviation of each feature
mean = X.mean(axis=0)
std = X.std(axis=0)

# Z-score normalization
X_norm = (X - mean) / std

print("Normalized X shape:", X_norm.shape)
print("Mean of first 5 features:", X_norm[:, :5].mean(axis=0))
print("Std of first 5 features:", X_norm[:, :5].std(axis=0))

Normalized X shape: (62, 2000)
Mean of first 5 features: [-8.77434326e-17 -3.58136460e-17 -3.58136460e-17 -3.67089871e-17
  4.07380223e-17]
Std of first 5 features: [1. 1. 1. 1. 1.]


In [10]:
#check any feature has sd as zero
print("Zero-variance features:", np.sum(std == 0))

Zero-variance features: 0


### Manifold Structure Learning (Laplacian Graph)

#### Calcilate pairwise distance

In [ ]:
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(X_norm, metric="euclidean")
#checking
print("Distance matrix shape:", distances.shape)
print("Distance between sample 0 and 1:", distances[0, 1])

Distance matrix shape: (62, 62)
Distance between sample 0 and 1: 48.70342714561361


In [ ]:
#itself
print("Distance between sample 0 and 1:", distances[0, 0])

Distance between sample 0 and 1: 0.0


#### Find the 5 nearest neighbours

In [ ]:
n_neighbors = 5

nearest_neighbors = np.argsort(distances, axis=1)[:, 1:n_neighbors + 1]
#chechikng
print("Nearest neighbours of sample 0:")
print(nearest_neighbors[0])

Nearest neighbours of sample 0:
[ 6  3 34 20  2]


paper doesn't mention about sigma value. We have to determine first

#### Determine Sigma

In [14]:
neighbor_distances = np.take_along_axis(
    distances,
    nearest_neighbors,
    axis=1
)

print("5-NN distances for first 5 samples:")
print(neighbor_distances[:5])

sigma = np.mean(neighbor_distances)

print("\nSigma:", sigma)

5-NN distances for first 5 samples:
[[42.23899689 43.84467105 44.38929656 44.58988379 46.97985307]
 [41.48249964 43.0260957  44.74213108 45.33946973 48.70342715]
 [39.32702424 42.42675832 43.08033261 44.36935275 44.55943621]
 [39.32702424 42.80143304 43.84467105 45.9949127  47.25845657]
 [35.63770338 38.26569602 40.33615602 40.60510398 41.28778414]]

Sigma: 43.70682015895078


collected the distance from every sample to its 5 nearest nodes. There were 310 distance total (62 * 5). Take their mean. Got sigma=43.7068...

#### Construct A

In [21]:
n_samples = X_norm.shape[0]

A = np.zeros((n_samples, n_samples))

for i in range(n_samples):
    for j in nearest_neighbors[i]:
        A[i, j] = np.exp(
            -(distances[i, j] ** 2) / (2 * sigma ** 2)
        )
# Make graph symmetric
A = np.maximum(A, A.T)

np.fill_diagonal(A, 0)

print("A shape:", A.shape)
print("Non-zero edges:", np.count_nonzero(A))

A shape: (62, 62)
Non-zero edges: 416


In [20]:
print(A)

[[0.         0.53748435 0.5611932  ... 0.         0.         0.        ]
 [0.53748435 0.         0.         ... 0.59216606 0.         0.        ]
 [0.5611932  0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.59216606 0.         ... 0.         0.         0.56472373]
 [0.         0.         0.         ... 0.         0.         0.57162179]
 [0.         0.         0.         ... 0.56472373 0.57162179 0.        ]]


#### Construct D and L_X

In [30]:
D = np.diag(A.sum(axis=1))

L_X = D - A

print("D shape:", D.shape)
print("L_X shape:", L_X.shape)

D shape: (62, 62)
L_X shape: (62, 62)


In [31]:
print(D)

[[3.96818631 0.         0.         ... 0.         0.         0.        ]
 [0.         2.96688244 0.         ... 0.         0.         0.        ]
 [0.         0.         4.75380633 ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 3.35820728 0.         0.        ]
 [0.         0.         0.         ... 0.         3.48106296 0.        ]
 [0.         0.         0.         ... 0.         0.         6.88017986]]


In [32]:
print("Laplacian symmetric:",
      np.allclose(L_X, L_X.T))

print("Diagonal of L_X (first 5):")
print(np.diag(L_X)[:5])

Laplacian symmetric: True
Diagonal of L_X (first 5):
[3.96818631 2.96688244 4.75380633 3.0229695  5.13400311]


### Maniflod preservation

#### Create a temporary latent representation

In [33]:
# Temporary latent representation
k = 10

np.random.seed(42)
Z = np.random.randn(k, X_norm.shape[0])

print("Z shape:", Z.shape)

Z shape: (10, 62)


#### Calculate the manifold-preservation term

In [34]:
manifold_term = np.trace(Z @ L_X @ Z.T)

print("Tr(Z L_X Z^T):", manifold_term)

Tr(Z L_X Z^T): 2337.6368658068573


#### Verification

In [35]:
pairwise_term = 0.0

for i in range(X_norm.shape[0]):
    for j in range(X_norm.shape[0]):
        zi = Z[:, i]
        zj = Z[:, j]
        pairwise_term += A[i, j] * np.sum((zi - zj) ** 2)

print("Trace form :", manifold_term)
print("Pairwise form:", pairwise_term)

Trace form : 2337.6368658068573
Pairwise form: 4675.273731613717


In [36]:
pairwise_term_half = 0.0

for i in range(X_norm.shape[0]):
    for j in range(i + 1, X_norm.shape[0]):
        zi = Z[:, i]
        zj = Z[:, j]

        pairwise_term_half += A[i, j] * np.sum((zi - zj) ** 2)

pairwise_term_half *= 2

print("Trace form :", manifold_term)
print("Pairwise form:", pairwise_term_half)

Trace form : 2337.6368658068573
Pairwise form: 4675.273731613712


In [37]:
pairwise_term = 0.0

for i in range(X_norm.shape[0]):
    for j in range(i + 1, X_norm.shape[0]):
        pairwise_term += A[i, j] * np.sum((Z[:, i] - Z[:, j]) ** 2)

print("Trace form:", manifold_term)
print("Pairwise form:", pairwise_term)

Trace form: 2337.6368658068573
Pairwise form: 2337.636865806856


For the symmetric graph laplacian L_X = D-A. \
The standard quadratic form identity is : \
Tr(Z L_X Z^T) = 1/2 Σ_ij A_ij ||z_i - z_j||²

In paper eqn number 1 omits the factor 1/2 in the displayed patwise formulation. In implementation therefore uses the matrix trace formuation directly.

### Sparse Manifold Autoencoder

#### Calculate the theoretical hidden dimension

In [38]:
n_samples = X_norm.shape[0]

c = np.sqrt(n_samples / 2)

print("Calculated number of clusters:", c)

Calculated number of clusters: 5.5677643628300215


#### Initialize Autoencoder Weights

In [39]:
# Autoencoder dimensions
k = 10

W1 = np.random.randn(X_norm.shape[1], k) * 0.01
W2 = np.random.randn(k, X_norm.shape[1]) * 0.01

print("W1 shape:", W1.shape)
print("W2 shape:", W2.shape)

W1 shape: (2000, 10)
W2 shape: (10, 2000)


#### Encoder: Generate Latent Representation

In [40]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Paper notation: X is features × samples
X_paper = X_norm.T

# Encoder
Z = sigmoid(W1.T @ X_paper)

print("X_paper shape:", X_paper.shape)
print("Latent representation Z shape:", Z.shape)
print("Z minimum:", Z.min())
print("Z maximum:", Z.max())

X_paper shape: (2000, 62)
Latent representation Z shape: (10, 62)
Z minimum: 0.22652149992229414
Z maximum: 0.8018092920693234


#### Decoder

In [41]:
X_hat = sigmoid(W2.T @ Z)

print("Reconstructed X shape:", X_hat.shape)
print("X_hat minimum:", X_hat.min())
print("X_hat maximum:", X_hat.max())

Reconstructed X shape: (2000, 62)
X_hat minimum: 0.4830330285247517
X_hat maximum: 0.5155584689995396


In [42]:
reconstruction_error = np.linalg.norm(X_paper - X_hat, 'fro')**2

print("Reconstruction error:", reconstruction_error)

Reconstruction error: 154990.39881519976


In [ ]:
row_l2_norms = np.linalg.norm(W1, axis=1)
l21_norm = np.sum(row_l2_norms)

print("Number of feature rows:", W1.shape[0])
print("L2,1 norm of W1:", l21_norm)
print("Minimum row L2 norm:", row_l2_norms.min())
print("Maximum row L2 norm:", row_l2_norms.max())

Number of feature rows: 2000
L2,1 norm of W1: 61.924346435353776
Minimum row L2 norm: 0.008943230027753967
Maximum row L2 norm: 0.05533519243474864


#### Feature wise sparsity score

In [47]:
feature_scores = np.linalg.norm(W1, axis=1)

print("Feature scores shape:", feature_scores.shape)
print("First 10 feature scores:")
print(feature_scores[:10])

Feature scores shape: (2000,)
First 10 feature scores:
[0.03345802 0.03632943 0.03629622 0.03559639 0.03462664 0.0393565
 0.02283248 0.02455538 0.0430988  0.01378374]


In [48]:
top_indices = np.argsort(feature_scores)[::-1][:10]

print("Top 10 feature indices:")
print(top_indices)

print("\nTop 10 feature scores:")
print(feature_scores[top_indices])

Top 10 feature indices:
[1522 1332  573  227 1823 1905 1403  357   54  401]

Top 10 feature scores:
[0.05533519 0.05490208 0.05358603 0.05324732 0.05229355 0.05223592
 0.05175862 0.05170411 0.05142761 0.0513203 ]


#### select top features

In [49]:
p = 100

selected_indices = np.argsort(feature_scores)[::-1][:p]

print("Number of selected features:", len(selected_indices))
print("Selected feature indices:")
print(selected_indices[:20])

Number of selected features: 100
Selected feature indices:
[1522 1332  573  227 1823 1905 1403  357   54  401 1493 1494  309 1676
 1454  336 1329  725  740  133]
